# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# TODO: 创建 w1, w2, b，以及包含三个参数的 SGD 优化器
# 按照提示，初始值都为 0.0，且需要梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
# 先把优化器建起来，初始学习率随便给（比如0.1），后面在循环里动态修改
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

for epoch in range(1, 2001):
    # TODO: 设置本轮学习率，并清空梯度
    # 根据提示：学习率 = 0.1 / (epoch ** 0.5)
    current_lr = 0.1 / (epoch ** 0.5)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr
        
    optimizer.zero_grad()
    
    # TODO: 调用 predict，同时使用 x1 和 x2
    y_pred = predict(x1, x2, w1, w2, b)
    
    # TODO: 计算逐样本绝对误差和平均损失
    sample_loss = torch.abs(y_pred - y)
    loss = sample_loss.mean()
    
    # TODO: 反向传播，更新参数
    loss.backward()
    optimizer.step()

# TODO: 在 torch.no_grad() 中，用训练后的参数重新计算预测与损失
with torch.no_grad():
    y_pred_final = predict(x1, x2, w1, w2, b)
    sample_losses_final = torch.abs(y_pred_final - y)
    mean_loss_final = sample_losses_final.mean()

# TODO: 打印三个参数、x1、x2、y、y_pred、逐样本损失、平均损失及其 shape
print(f"w1: {w1.item():.4f}, w2: {w2.item():.4f}, b: {b.item():.4f}")
print(f"x1: {x1}, shape: {x1.shape}")
print(f"x2: {x2}, shape: {x2.shape}")
print(f"y: {y}, shape: {y.shape}")
print(f"y_pred: {y_pred_final}, shape: {y_pred_final.shape}")
print(f"逐样本损失: {sample_losses_final}, shape: {sample_losses_final.shape}")
print(f"平均损失: {mean_loss_final.item():.4f}, shape: {mean_loss_final.shape}")


w1: 1.9990, w2: -3.0010, b: 0.9994
x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]), shape: torch.Size([6])
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]), shape: torch.Size([6])
y: tensor([-6., -1., -5.,  6., -4.,  4.]), shape: torch.Size([6])
y_pred: tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954]), shape: torch.Size([6])
逐样本损失: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]), shape: torch.Size([6])
平均损失: 0.0024, shape: torch.Size([])


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

1. 一次预测需要 2 个输入张量（即 x1 和 x2）。x1 和 x2 是 shape 为 (6,) 的张量。这里的 6 代表样本的数量，它们并不是 6 个不同的张量。x1 包含了 6 个样本的第一个特征，x2 包含了 6 个样本的第二个特征。它们是一一对应的关系（如 x1[0] 和 x2[0] 共同组成第 0 个样本）
2. 预测公式：加入了第二个输入，从 w*x+b 变成了 w1*x1 + w2*x2 + b。
· 参数列表：多了一个权重参数，从 [w, b] 变成了 [w1, w2, b]。
· 损失计算：从上周的均方误差（平方后求均值，MSE）改为了绝对误差（先求绝对值再求均值，MAE）。
· 优化器步骤：基本步骤没有变。依然是：清空梯度 (zero_grad) -> 反向传播 (backward) -> 更新参数 (step)。只是这次增加了一步在循环里动态调整学习率的操作
3. 最终参数：w1 ≈ 1.9990，w2 ≈ -3.0010，b ≈ 0.9994。
· 平均损失：0.0024。
· 判断依据：随着训练轮数增加，平均损失（Loss）持续下降并最终趋于极小值，且模型输出的预测值 y_pred 与真实目标值 y 极度吻合（差值极小），说明模型成功学习到了数据的规律，训练是有效的
4. 因为这是一个线性回归模型，只要参数开启了梯度跟踪（requires_grad=True），PyTorch 会根据公式计算偏导数。参数对损失的梯度依赖于输入特征（如 x_1, x_2），只要输入特征不为 0，梯度就不会为 0，优化器就可以通过梯度下降来更新参数。这不像神经网络中的神经元，如果不加随机初始化全设为 0 会导致对称性无法打破的问题
